# 经典机器学习情感分类介绍

本 notebook 介绍文本表示、特征处理与经典分类器，及其调用接口。实验使用 MARC 中文商品评论，标签为负面0、正面1。

训练集20,000条，正负各10,000条；验证集4,000条，正负各2,000条，与预训练模型实验使用相同划分。训练集用于训练，验证集用于验证与模型选择。

第1—3节介绍方法，第4—8节包括环境与数据读取代码，以及文本表示、分类器调用与评价说明，第9节说明实验结果。


## 1 文本怎样变成分类器的输入

向量空间模型把一条评论表示为一行数值，每一列对应一个词项。

| 表示 | 含义 | 对评论的作用 |
|---|---|---|
| 词袋／词频 | 统计词项出现次数 | 表示文本含有哪些词项及其频次 |
| TF-IDF | 结合词频和文档频率赋权 | 减小普遍出现词项的权重 |
| n-gram | 使用相邻字符或词的组合 | 保留局部组合，例如“不好” |

中文可先分词后使用词级特征，也可使用字符特征。n-gram 表示局部组合。词级表示需要先对中文文本进行分词。

词频记录词项出现次数；TF-IDF 结合词项在不同文档中的分布进行赋权。

## 2 特征选择与降维

| 方法 | 做了什么 | 常见用途 |
|---|---|---|
| 卡方特征选择 | 保留与类别关联较强的一部分原词项 | 保留部分原始特征 |
| TruncatedSVD | 将高维特征组合成较低维的连续向量 | 获得低维表示 |
| L2归一化 | 统一向量长度 | 距离与相似度比较 |
| 标准化 | 调整各维的尺度 | 统一特征尺度 |

特征选择仍保留原词项；SVD产生新的组合维度。处理器只在训练集调用fit，验证集调用transform。多项式和补集朴素贝叶斯使用非负词频或TF-IDF输入。

## 3 生成式与判别式分类

教材以朴素贝叶斯说明生成式分类：学习类别先验及类条件特征分布，再推断评论的类别。Logistic回归、SVM等判别式方法直接学习类别条件概率或分类边界。

sklearn统一使用fit学习、predict分类，算法内部的学习方式则不同。下面按原理分组介绍15项实现。

### 3.1 朴素贝叶斯

给定类别后，假设各特征条件独立，用词项统计计算类别分数，选择分数最高的类别。

| sklearn类 | 输入与区别 | 主要参数 |
|---|---|---|
| MultinomialNB | 非负词频或TF-IDF；利用词项权重统计 | alpha：平滑强度 |
| ComplementNB | 利用其他类别的词项统计构造分数 | alpha：平滑强度 |
| BernoulliNB | 词项出现／未出现的0、1表示 | binarize：二值化阈值；alpha |

平滑用于避免未出现的词项得到零概率。TF-IDF配合多项式朴素贝叶斯是常用实践，其输入含义与原始计数仍有区别。

### 3.2 线性分类方法

学习每个特征的权重，把评论向量加权组合成分类分数。它们可直接处理高维稀疏TF-IDF。

| sklearn类 | 分类思路 | 主要参数 |
|---|---|---|
| LogisticRegression | 将线性分数转换为类别概率 | C越小，正则化越强；max_iter |
| LinearSVC | 学习线性最大间隔边界 | C控制间隔与分类错误的权衡 |
| RidgeClassifier | 用平方损失学习类别编码，并限制权重大小 | alpha越大，正则化越强 |
| Perceptron | 对误分类样本迭代修正权重 | max_iter、tol |

LogisticRegression输出类别概率；LinearSVC输出带符号的决策分数，表示边界两侧的类别倾向。

### 3.3 距离与核方法

| sklearn类 | 分类思路 | 主要参数 |
|---|---|---|
| NearestCentroid | 计算各类中心，选择最近的中心 | 距离度量与输入表示 |
| KNeighborsClassifier | 找到近邻评论，根据标签投票 | n_neighbors：邻居数；weights：投票权重；metric |
| SVC（RBF） | 用核函数学习非线性分类边界 | C、gamma：样本影响范围 |

距离计算受文本表示方式影响。KNN主要在预测时搜索近邻；核SVM训练成本也受样本数量影响。

### 3.4 决策树与集成方法

决策树按特征阈值逐层划分样本；集成方法组合多棵树或多个弱分类器。

| sklearn类 | 分类思路 | 主要参数 |
|---|---|---|
| DecisionTreeClassifier | 沿分支到达叶节点，输出类别 | max_depth、min_samples_leaf |
| RandomForestClassifier | 多棵随机化树共同投票 | n_estimators、max_depth |
| AdaBoostClassifier | 逐轮增加误分类样本权重，组合弱分类器 | n_estimators、learning_rate |
| GradientBoostingClassifier | 沿损失下降方向逐轮增加树，修正预测 | n_estimators、learning_rate、max_depth |

树深度和叶节点样本数影响模型复杂度。

### 3.5 多层感知机

MLPClassifier 使用隐藏层进行非线性变换，通过反向传播学习网络权重。

主要参数包括 hidden_layer_sizes（隐藏层数量与每层单元数）、max_iter（最多训练轮数）和 early_stopping（是否使用内部留出数据进行早停）。启用内部早停时，scikit-learn 使用 Accuracy 监测内部留出数据；本实验仍在统一的验证集上报告评价结果。


### 3.6 方法汇总与实验要求

**以上15种方法均需完成实验，不是选做。** 各方法使用相同的训练集与验证集，记录实际采用的文本表示、特征处理、分类器配置及评价结果，并在报告中汇总比较。

| 方法组 | 需完成的方法 | 数量 |
|---|---|---|
| 朴素贝叶斯 | MultinomialNB、ComplementNB、BernoulliNB | 3 |
| 线性分类 | LogisticRegression、LinearSVC、RidgeClassifier、Perceptron | 4 |
| 距离与核方法 | NearestCentroid、KNeighborsClassifier、SVC（RBF） | 3 |
| 决策树与集成 | DecisionTreeClassifier、RandomForestClassifier、AdaBoostClassifier、GradientBoostingClassifier | 4 |
| 多层感知机 | MLPClassifier | 1 |
| 合计 | 全部方法均需完成 | 15 |


## 4 环境与目录

ROOT 指向包含 data 文件夹的教学包目录。第4—5节提供环境检查与数据读取代码，第6—8节介绍文本表示、分类器调用与评价。


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
import sklearn

ROOT = Path.cwd()  # 修改为教学包目录
DATA = ROOT / "data"
print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("数据目录:", DATA.resolve())


Python: 3.11.14 (main, Dec  2 2025, 19:48:49) [Clang 21.1.4 ]
scikit-learn: 1.5.2
数据目录: /home/biscuit/Documents/HIT-NLP/lab1/实验一情感分析教学包/data


## 5 读取训练集与验证集

CSV 包含 id、text、label。评分1—2星对应负面0，4—5星对应正面1，不使用3星样本。评分标签与文本情感可能存在差异。

本节检查数据字段、标签、空文本和样本 ID，并显示数量及示例。


In [2]:
def read_split(name):
    assert name in {"train", "validation"}
    frame = pd.read_csv(
        DATA / f"{name}.csv", keep_default_na=False,
        dtype={"id": str, "text": str},
    )
    assert {"id", "text", "label"}.issubset(frame.columns)
    assert len(frame) > 0 and frame["label"].isin([0, 1]).all()
    assert not frame["text"].str.strip().eq("").any()
    assert not frame["id"].str.strip().eq("").any()
    assert not frame["id"].duplicated().any()
    frame["label"] = frame["label"].astype(int)
    return frame

train_df = read_split("train")
validation_df = read_split("validation")
assert not set(train_df["id"]).intersection(validation_df["id"])
for name, frame in [("train", train_df), ("validation", validation_df)]:
    print(name, "样本数:", len(frame), "标签分布:", frame["label"].value_counts().to_dict())
display(train_df.head())


train 样本数: 20000 标签分布: {1: 10000, 0: 10000}
validation 样本数: 4000 标签分布: {0: 2000, 1: 2000}


,id,text,label
0,zh_0826191,饭盒还算可以，但布袋质量差，还没用拉链就坏了。,1
1,zh_0978217,一天到晚吸毒、作画什么的，神经病似的的艺术家？！,0
2,zh_0245009,都是很基础的摄影，网上到处都看得见，编在书里毫无价值。不建议购买，！,0
3,zh_0615419,刚开始用的前几次，洗完头头发很顺；用到一半的时候就出问题了，每次冲洗护发素的时候能明显地感觉...,0
4,zh_0678062,拿到了吓一跳，椭圆形的，长直径刚好跟一元硬币一样。。。。。。。。。,1


## 6 文本表示与特征处理

文本转换器将评论转换为数值特征矩阵，每行对应一条评论，每列对应一个特征。
- CountVectorizer 生成词频表示
- TfidfVectorizer 生成 TF-IDF 表示。

fit_transform 在训练文本上学习表示并生成特征矩阵；transform 使用已经学习的表示处理验证文本。训练与验证特征的列应保持一致。

特征选择、降维等处理属于文本表示流程，其输入和输出应与分类器要求对应。各实验记录实际采用的表示与处理配置。


In [3]:
# 实验公共工具：随机种子、评价指标与结果保存（与微调路线使用统一的评价口径）。
# 全部实验产物保存到教学包内的 classical/ 目录（预测明细 preds/、模型 models/）。
import json
import random

import numpy as np

SEED = 42                        # 全局随机种子（固定以保证结果可复现）
CLASS_NAMES = ["负面", "正面"]    # 0 = 负面（1-2 星），1 = 正面（4-5 星）

SAVE_DIR = ROOT / "classical"    # 实验产物目录
(SAVE_DIR / "preds").mkdir(parents=True, exist_ok=True)
(SAVE_DIR / "models").mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    """统一设置 random / numpy 随机种子（每个方法/配置前调用，保证公平）。"""
    random.seed(seed)
    np.random.seed(seed)


def compute_metrics(y_true, y_pred):
    """统一评价指标：Accuracy、Macro-F1、宏平均 Precision / Recall。"""
    from sklearn.metrics import (
        accuracy_score, f1_score, precision_score, recall_score)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "precision_macro": float(
            precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(
            recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def save_json(obj, path):
    """保存 JSON（ensure_ascii=False，保留中文可读性）。"""
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2, default=str)
    print(f"[结果] 已保存: {Path(path).relative_to(ROOT)}")


print(f"公共工具就绪：SEED={SEED}；产物目录：{SAVE_DIR.name}/")


公共工具就绪：SEED=42；产物目录：classical/


In [ ]:
# 文本表示对照实验：固定逻辑回归探针（C=10，收敛稳定、速度快），只改变
# “文本 → 特征”的表示方式，在验证集上比较 Macro-F1，选出 15 种分类器
# 主实验的统一文本表示。六个候选配置共享基础参数：sublinear_tf 抑制高频
# 词项、min_df=2 过滤低频特征、L2 归一化。
import time

import jieba
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion


def jieba_cut(text):
    """jieba 分词（词级表示的 tokenizer）；中文无天然空格分词。"""
    return [tok for tok in jieba.lcut(text) if tok.strip()]

# 基础参数 sublinear_tf表示高频词项抑制、min_df=2过滤低频特征、L2归一化
_BASE_TFIDF = dict(sublinear_tf=True, min_df=2, norm="l2")

# 六个候选文本表示配置（字符级 n-gram、jieba 词级 n-gram、两者拼接）
# key : 配置名，value : TfidfVectorizer 参数字典或 FeatureUnion 配置
REPR_CONFIGS = {
    "char12": dict(analyzer="char", ngram_range=(1, 2), **_BASE_TFIDF),
    "char13": dict(analyzer="char", ngram_range=(1, 3), **_BASE_TFIDF),
    "char14": dict(analyzer="char", ngram_range=(1, 4), max_features=600_000,
                   **_BASE_TFIDF),
    "word1": dict(tokenizer=jieba_cut, token_pattern=None, ngram_range=(1, 1),
                  **_BASE_TFIDF),
    "word12": dict(tokenizer=jieba_cut, token_pattern=None, ngram_range=(1, 2),
                   **_BASE_TFIDF),
    "char13_word12": dict(union=[("char13", "char13"), ("word12", "word12")]),
}
REPR_NOTES = {
    "char12": "字符 1-2 gram",
    "char13": "字符 1-3 gram",
    "char14": "字符 1-4 gram（≤60万特征）",
    "word1": "jieba 词 1-gram",
    "word12": "jieba 词 1-2 gram",
    "char13_word12": "字符 1-3 gram + jieba 词 1-2 gram 拼接",
}


def build_vectorizer(name, **overrides):
    """按配置名name构造（未拟合）的 TF-IDF 向量器；overrides 用于单因素改进对照。"""
    params = dict(REPR_CONFIGS[name]) # 获得给tfidf的参数字典
    params.pop("union", None) # 如果是拼接的配置，去掉union参数
    params.update(overrides) # 如果有额外的参数覆盖，更新参数字典
    return TfidfVectorizer(**params) 


def build_union_vectorizer():
    """词级 + 字符级拼接（FeatureUnion），兼取两种表示视角。"""
    return FeatureUnion([("char13", build_vectorizer("char13")),
                         ("word12", build_vectorizer("word12"))])


train_texts = train_df["text"].tolist()
y_train = train_df["label"].to_numpy()
dev_texts = validation_df["text"].tolist()
y_dev = validation_df["label"].to_numpy()

probe_rows = []
for name in REPR_NOTES:   # char12 → char13 → char14 → word1 → word12 → 词字拼接
    set_seed()            # 每种配置重新固定种子，保证对照公平
    t0 = time.time()
    vec = build_union_vectorizer() if name == "char13_word12" else build_vectorizer(name) # 文本2特征矩阵构造器

    X_tr = vec.fit_transform(train_texts) # 调用sklearn的fit_transform方法，学习表示并生成训练特征矩阵
    X_va = vec.transform(dev_texts) # 调用sklearn的transform方法，使用已经学习的表示处理验证文本
    vec_sec = time.time() - t0
    t1 = time.time()
    clf = LogisticRegression(C=10.0, max_iter=1000, solver="lbfgs",
                             random_state=SEED)
    clf.fit(X_tr, y_train)
    pred = clf.predict(X_va)
    run_sec = time.time() - t1
    m = compute_metrics(y_dev, pred)
    probe_rows.append({
        "配置": name, "说明": REPR_NOTES[name], "特征维度": X_tr.shape[1],
        "训练非零元": int(X_tr.nnz), "转换耗时(s)": round(vec_sec, 1),
        "训练+预测耗时(s)": round(run_sec, 1), **m,
    })
    print(f"[{name}] 维度 {X_tr.shape[1]}，Macro-F1={m['macro_f1']:.4f}，"
          f"Accuracy={m['accuracy']:.4f}")

probe_df = pd.DataFrame(probe_rows).sort_values("macro_f1", ascending=False)
char_best = probe_df[probe_df["配置"].str.startswith("char")].iloc[0]
word_best = probe_df[probe_df["配置"].str.startswith("word")].iloc[0]
print(f"结论：{probe_df.iloc[0]['配置']} 验证 Macro-F1 最高"
      f"（{probe_df.iloc[0]['macro_f1']:.4f}），选为 15 种分类器主实验的统一文本表示；"
      f"字符级最佳 {char_best['配置']}（{char_best['macro_f1']:.4f}）优于词级最佳 "
      f"{word_best['配置']}（{word_best['macro_f1']:.4f}）")

probe_df.to_csv(SAVE_DIR / "repr_comparison.csv", index=False, encoding="utf-8-sig")
save_json({"selected": str(probe_df.iloc[0]["配置"]),
           "val_macro_f1": float(probe_df.iloc[0]["macro_f1"])},
          SAVE_DIR / "repr_choice.json")
display(probe_df)


[char12] 维度 66343，Macro-F1=0.8310，Accuracy=0.8310
[char13] 维度 172914，Macro-F1=0.8442，Accuracy=0.8442


Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache


[char14] 维度 255232，Macro-F1=0.8440，Accuracy=0.8440


Loading model cost 0.273 seconds.
Prefix dict has been built successfully.


[word1] 维度 16158，Macro-F1=0.8225，Accuracy=0.8225
[word12] 维度 72406，Macro-F1=0.8407，Accuracy=0.8407
[char13_word12] 维度 245320，Macro-F1=0.8437，Accuracy=0.8438
结论：char13 验证 Macro-F1 最高（0.8442），选为 15 种分类器主实验的统一文本表示；字符级最佳 char13（0.8442）优于词级最佳 word12（0.8407）
[结果] 已保存: classical/repr_choice.json


,配置,说明,特征维度,训练非零元,转换耗时(s),训练+预测耗时(s),accuracy,macro_f1,precision_macro,recall_macro
1,char13,字符 1-3 gram,172914,2225990,1.4,0.6,0.84425,0.844223,0.844492,0.84425
2,char14,字符 1-4 gram（≤60万特征）,255232,2546743,3.8,1.1,0.84400,0.843972,0.844251,0.84400
5,char13_word12,字符 1-3 gram + jieba 词 1-2 gram 拼接,245320,3137073,3.9,1.3,0.84375,0.843734,0.843895,0.84375
4,word12,jieba 词 1-2 gram,72406,911083,2.6,0.6,0.84075,0.840742,0.840822,0.84075
0,char12,字符 1-2 gram,66343,1617759,0.6,0.7,0.83100,0.830964,0.831279,0.83100
3,word1,jieba 词 1-gram,16158,488414,2.4,2.1,0.82250,0.822496,0.822526,0.82250


In [5]:
# 统一文本表示：char13（字符 1-3 gram TF-IDF）——探针实验中验证集 Macro-F1 最高，
# 且字符级表示对分词错误不敏感。后续 15 种分类器共用该特征矩阵。
REPR_NAME = "char13"
vectorizer = build_vectorizer(REPR_NAME)
X_train = vectorizer.fit_transform(train_texts)   # 训练集：学习词表并生成特征
X_dev = vectorizer.transform(dev_texts)           # 验证集：用同一词表转换（列一致）
print(f"统一表示：{REPR_NAME}（{REPR_NOTES[REPR_NAME]}）")
print(f"特征矩阵：train {X_train.shape}（非零元 {X_train.nnz}），dev {X_dev.shape}")


统一表示：char13（字符 1-3 gram）
特征矩阵：train (20000, 172914)（非零元 2225990），dev (4000, 172914)


## 7 分类器训练与预测

分类器的 fit 接收训练特征及对应标签，predict 对验证特征输出预测标签。文本转换器、特征处理器与分类器共同构成评论分类流程。

分别完成第3节列出的15种分类方法，记录各方法的实际配置与运行结果。不同分类器的输入条件和参数含义见前文介绍。


In [6]:
# 15 种分类方法规格（指导书第 3 节：以上 15 种方法均需完成实验，不是选做）。
# 每个方法记录：key（结果文件短名）、name_cn（名称）、group（方法组）、
# factory（构造未训练的分类器）、params_note（主要参数说明，写入结果表）。
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron, RidgeClassifier
from sklearn.naive_bayes import BernoulliNB, ComplementNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

SVD_DIM = 300   # MLP 前的 TruncatedSVD 降维维度（sklearn 的 MLP 不支持稀疏输入）

CLASSICAL_METHODS = [
    # ---- 3.1 朴素贝叶斯（生成式，利用词项统计）----
    dict(key="multinomial_nb", name_cn="MultinomialNB", group="朴素贝叶斯",
         factory=lambda: MultinomialNB(alpha=1.0),
         params_note="alpha=1.0（拉普拉斯平滑）；输入为 TF-IDF 非负权重"),
    dict(key="complement_nb", name_cn="ComplementNB", group="朴素贝叶斯",
         factory=lambda: ComplementNB(alpha=1.0),
         params_note="alpha=1.0；用互补类统计构造权重，缓解类别不平衡"),
    dict(key="bernoulli_nb", name_cn="BernoulliNB", group="朴素贝叶斯",
         factory=lambda: BernoulliNB(alpha=1.0, binarize=0.0),
         params_note="alpha=1.0, binarize=0.0（TF-IDF>0 视为词项出现）"),
    # ---- 3.2 线性分类方法（判别式，直接学习特征权重）----
    dict(key="logistic_regression", name_cn="LogisticRegression", group="线性分类",
         factory=lambda: LogisticRegression(C=1.0, max_iter=1000,
                                            solver="lbfgs", random_state=SEED),
         params_note="C=1.0（L2 正则强度倒数）, max_iter=1000, lbfgs"),
    dict(key="linear_svc", name_cn="LinearSVC", group="线性分类",
         factory=lambda: LinearSVC(C=1.0, max_iter=5000, random_state=SEED),
         params_note="C=1.0（间隔与错误的权衡）"),
    dict(key="ridge", name_cn="RidgeClassifier", group="线性分类",
         factory=lambda: RidgeClassifier(alpha=1.0, random_state=SEED),
         params_note="alpha=1.0（L2 正则强度）"),
    dict(key="perceptron", name_cn="Perceptron", group="线性分类",
         factory=lambda: Perceptron(max_iter=1000, tol=1e-3, random_state=SEED),
         params_note="max_iter=1000, tol=1e-3（误分类驱动迭代更新）"),
    # ---- 3.3 距离与核方法 ----
    dict(key="nearest_centroid", name_cn="NearestCentroid", group="距离与核方法",
         factory=lambda: NearestCentroid(metric="euclidean"),
         params_note="欧氏距离；TF-IDF 已 L2 归一化，距离序等价余弦相似度序"),
    dict(key="knn", name_cn="KNeighborsClassifier", group="距离与核方法",
         factory=lambda: KNeighborsClassifier(
             n_neighbors=15, weights="distance", metric="euclidean",
             algorithm="brute", n_jobs=-1),
         params_note="k=15, weights=distance, 暴力搜索欧氏距离"),
    dict(key="svc_rbf", name_cn="SVC(RBF)", group="距离与核方法",
         factory=lambda: SVC(C=1.0, kernel="rbf", gamma="scale",
                             cache_size=1024, random_state=SEED),
         params_note="C=1.0, gamma=scale, kernel=rbf, cache=1GB"),
    # ---- 3.4 决策树与集成方法 ----
    dict(key="decision_tree", name_cn="DecisionTreeClassifier", group="决策树与集成",
         factory=lambda: DecisionTreeClassifier(random_state=SEED),
         params_note="默认完全生长（max_depth=None）"),
    dict(key="random_forest", name_cn="RandomForestClassifier", group="决策树与集成",
         factory=lambda: RandomForestClassifier(
             n_estimators=200, n_jobs=-1, random_state=SEED),
         params_note="n_estimators=200, max_features=sqrt（默认）, 并行"),
    dict(key="adaboost", name_cn="AdaBoostClassifier", group="决策树与集成",
         factory=lambda: AdaBoostClassifier(
             n_estimators=200, learning_rate=1.0, random_state=SEED),
         params_note="n_estimators=200, learning_rate=1.0, 基学习器为深度1决策树"),
    dict(key="gradient_boosting", name_cn="GradientBoostingClassifier",
         group="决策树与集成",
         factory=lambda: GradientBoostingClassifier(
             n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED),
         params_note="n_estimators=200, learning_rate=0.1, max_depth=3"),
    # ---- 3.5 多层感知机 ----
    dict(key="mlp", name_cn="MLPClassifier", group="多层感知机",
         factory=lambda: MLPClassifier(
             hidden_layer_sizes=(256,), activation="relu", alpha=1e-4,
             max_iter=60, early_stopping=True, n_iter_no_change=5,
             random_state=SEED),
         needs_svd=True,
         params_note=f"TruncatedSVD({SVD_DIM})→隐层(256), relu, 内部早停, max_iter=60"),
]

assert len(CLASSICAL_METHODS) == 15
for spec in CLASSICAL_METHODS:
    print(f"[{spec['group']}] {spec['name_cn']}")
print(f"共 {len(CLASSICAL_METHODS)} 种方法（朴素贝叶斯 3 + 线性分类 4 + "
      f"距离与核方法 3 + 决策树与集成 4 + 多层感知机 1）")


[朴素贝叶斯] MultinomialNB
[朴素贝叶斯] ComplementNB
[朴素贝叶斯] BernoulliNB
[线性分类] LogisticRegression
[线性分类] LinearSVC
[线性分类] RidgeClassifier
[线性分类] Perceptron
[距离与核方法] NearestCentroid
[距离与核方法] KNeighborsClassifier
[距离与核方法] SVC(RBF)
[决策树与集成] DecisionTreeClassifier
[决策树与集成] RandomForestClassifier
[决策树与集成] AdaBoostClassifier
[决策树与集成] GradientBoostingClassifier
[多层感知机] MLPClassifier
共 15 种方法（朴素贝叶斯 3 + 线性分类 4 + 距离与核方法 3 + 决策树与集成 4 + 多层感知机 1）


In [7]:
# 分类器管线与评价工具：多数方法直接使用稀疏 TF-IDF 特征；
# MLP 需要稠密输入，按指导书第 2 节先用 TruncatedSVD 降至 300 维
# （只在训练集上 fit，防止信息泄漏）。
# KNN 在 L2 归一化 TF-IDF 上使用欧氏距离——对单位向量，
# 欧氏距离的排序与余弦相似度完全一致。
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline


def build_pipeline(spec):
    """构造分类器管线（含 MLP 的 SVD 降维步骤）。"""
    clf = spec["factory"]()
    if spec.get("needs_svd"):
        return Pipeline([
            ("svd", TruncatedSVD(n_components=SVD_DIM, random_state=SEED)),
            ("clf", clf),
        ])
    return Pipeline([("clf", clf)])


def predict_scores(pipeline, X):
    """提取“正面(1)”分数（概率或决策函数值），用于预测明细；最近质心无分数输出。"""
    clf = pipeline[-1]
    if hasattr(clf, "predict_proba"):
        return pipeline.predict_proba(X)[:, 1]
    if hasattr(clf, "decision_function"):
        return pipeline.decision_function(X)
    return np.full(X.shape[0], np.nan)


In [8]:
# 逐一训练全部 15 种方法：fit(训练特征, 训练标签) → predict(验证特征) → 统一评价。
# 每个方法前重置随机种子（含 MLP 的 SVD 与集成算法的随机性），保证公平与可复现。
# 耗时列为墙钟时间、仅供参照（与硬件与后台负载相关）。
# 说明：RBF SVC 与梯度提升较慢，本节约运行 8—10 分钟。
import time

method_results = []
for spec in CLASSICAL_METHODS:
    print(f"===== [{spec['group']}] {spec['name_cn']} =====", flush=True)
    set_seed()
    t_fit = time.time()
    pipe = build_pipeline(spec)
    pipe.fit(X_train, y_train)
    fit_sec = time.time() - t_fit
    t_pred = time.time()
    pred = pipe.predict(X_dev)
    scores = predict_scores(pipe, X_dev)
    pred_sec = time.time() - t_pred
    metrics = compute_metrics(y_dev, pred)
    method_results.append({
        "spec": spec, "pipe": pipe, "pred": pred, "scores": scores,
        "metrics": metrics,
        "cm": confusion_matrix(y_dev, pred, labels=[0, 1]),
        "report": classification_report(y_dev, pred, labels=[0, 1],
                                        target_names=CLASS_NAMES,
                                        output_dict=True, zero_division=0),
        "fit_sec": fit_sec, "pred_sec": pred_sec,
    })
    print(f"  Accuracy={metrics['accuracy']:.4f}  Macro-F1={metrics['macro_f1']:.4f}"
          f"  P={metrics['precision_macro']:.4f}  R={metrics['recall_macro']:.4f}"
          f"（训练 {fit_sec:.0f}s / 预测 {pred_sec:.0f}s）", flush=True)

print(f"\n15 种方法全部完成（累计训练+预测 "
      f"{sum(r['fit_sec'] + r['pred_sec'] for r in method_results):.0f}s）")


===== [朴素贝叶斯] MultinomialNB =====
  Accuracy=0.8280  Macro-F1=0.8280  P=0.8281  R=0.8280（训练 0s / 预测 0s）
===== [朴素贝叶斯] ComplementNB =====
  Accuracy=0.8280  Macro-F1=0.8280  P=0.8281  R=0.8280（训练 0s / 预测 0s）
===== [朴素贝叶斯] BernoulliNB =====
  Accuracy=0.7987  Macro-F1=0.7970  P=0.8095  R=0.7988（训练 0s / 预测 0s）
===== [线性分类] LogisticRegression =====
  Accuracy=0.8400  Macro-F1=0.8400  P=0.8401  R=0.8400（训练 0s / 预测 0s）
===== [线性分类] LinearSVC =====
  Accuracy=0.8442  Macro-F1=0.8442  P=0.8445  R=0.8442（训练 0s / 预测 0s）
===== [线性分类] RidgeClassifier =====
  Accuracy=0.8468  Macro-F1=0.8467  P=0.8471  R=0.8468（训练 0s / 预测 0s）
===== [线性分类] Perceptron =====
  Accuracy=0.8147  Macro-F1=0.8145  P=0.8164  R=0.8148（训练 0s / 预测 0s）
===== [距离与核方法] NearestCentroid =====
  Accuracy=0.7895  Macro-F1=0.7895  P=0.7896  R=0.7895（训练 0s / 预测 0s）
===== [距离与核方法] KNeighborsClassifier =====
  Accuracy=0.7808  Macro-F1=0.7805  P=0.7822  R=0.7808（训练 0s / 预测 4s）
===== [距离与核方法] SVC(RBF) =====
  Accuracy=0.8450  Macro-F1=0.

/home/biscuit/Documents/HIT-NLP/lab1/.venv/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


  Accuracy=0.8195  Macro-F1=0.8195  P=0.8196  R=0.8195（训练 39s / 预测 0s）
===== [决策树与集成] GradientBoostingClassifier =====
  Accuracy=0.8125  Macro-F1=0.8125  P=0.8127  R=0.8125（训练 100s / 预测 0s）
===== [多层感知机] MLPClassifier =====
  Accuracy=0.8320  Macro-F1=0.8320  P=0.8321  R=0.8320（训练 22s / 预测 0s）

15 种方法全部完成（累计训练+预测 413s）


## 8 分类评价

使用同一验证集评价全部15种方法，报告 Accuracy、Macro-F1、Precision 和 Recall。

| 指标 | 含义 |
|---|---|
| Accuracy | 预测正确的样本比例 |
| Precision | 被预测为某类的样本中，真正属于该类的比例 |
| Recall | 某类真实样本中，被正确识别的比例 |
| F1 | Precision 与 Recall 的调和平均值 |
| Macro-F1 | 正负两类 F1 的平均值 |

分类报告按负面0、正面1展示。混淆矩阵的行表示真实类别，列表示预测类别。汇总结果时注明各项指标的类别或平均方式，保持各方法的评价口径一致。


In [9]:
# 指标汇总表：与报告 4.1 节表 1 同口径（Accuracy / Macro-F1 / 宏平均 P、R），
# 按 Macro-F1 降序排列。
result_df = pd.DataFrame([{
    "方法": r["spec"]["name_cn"], "方法组": r["spec"]["group"],
    "key": r["spec"]["key"],
    "Accuracy": r["metrics"]["accuracy"],
    "Macro-F1": r["metrics"]["macro_f1"],
    "Precision(macro)": r["metrics"]["precision_macro"],
    "Recall(macro)": r["metrics"]["recall_macro"],
    "训练耗时(s)": round(r["fit_sec"], 1),
    "预测耗时(s)": round(r["pred_sec"], 1),
    "参数配置": r["spec"]["params_note"],
} for r in method_results]).sort_values("Macro-F1", ascending=False)
best_row = result_df.iloc[0]
print(f"最佳方法：{best_row['方法']}（Macro-F1={best_row['Macro-F1']:.4f}，"
      f"Accuracy={best_row['Accuracy']:.4f}）")
display(result_df)


最佳方法：RidgeClassifier（Macro-F1=0.8467，Accuracy=0.8468）


,方法,方法组,key,Accuracy,Macro-F1,Precision(macro),Recall(macro),训练耗时(s),预测耗时(s),参数配置
5,RidgeClassifier,线性分类,ridge,0.84675,0.846712,0.847094,0.84675,0.1,0.0,alpha=1.0（L2 正则强度）
9,SVC(RBF),距离与核方法,svc_rbf,0.84500,0.844981,0.845167,0.84500,174.9,46.1,"C=1.0, gamma=scale, kernel=rbf, cache=1GB"
4,LinearSVC,线性分类,linear_svc,0.84425,0.844221,0.844511,0.84425,0.3,0.0,C=1.0（间隔与错误的权衡）
3,LogisticRegression,线性分类,logistic_regression,0.84000,0.839990,0.840087,0.84000,0.4,0.0,"C=1.0（L2 正则强度倒数）, max_iter=1000, lbfgs"
14,MLPClassifier,多层感知机,mlp,0.83200,0.831993,0.832056,0.83200,21.7,0.4,"TruncatedSVD(300)→隐层(256), relu, 内部早停, max_ite..."
0,MultinomialNB,朴素贝叶斯,multinomial_nb,0.82800,0.827988,0.828095,0.82800,0.0,0.0,alpha=1.0（拉普拉斯平滑）；输入为 TF-IDF 非负权重
1,ComplementNB,朴素贝叶斯,complement_nb,0.82800,0.827988,0.828095,0.82800,0.0,0.0,alpha=1.0；用互补类统计构造权重，缓解类别不平衡
11,RandomForestClassifier,决策树与集成,random_forest,0.81950,0.819500,0.819501,0.81950,8.9,0.2,"n_estimators=200, max_features=sqrt（默认）, 并行"
12,AdaBoostClassifier,决策树与集成,adaboost,0.81950,0.819488,0.819582,0.81950,38.8,0.5,"n_estimators=200, learning_rate=1.0, 基学习器为深度1决策树"
6,Perceptron,线性分类,perceptron,0.81475,0.814513,0.816367,0.81475,0.0,0.0,"max_iter=1000, tol=1e-3（误分类驱动迭代更新）"


In [10]:
# 分类报告与混淆矩阵（逐一展示；完整报告随后保存为 JSON）。
# 分类报告按 负面(0) / 正面(1) 分列 P / R / F1；
# 混淆矩阵的行为真实类别、列为预测类别。
for r in method_results:
    spec = r["spec"]
    print(f"\n===== [{spec['group']}] {spec['name_cn']} =====")
    display(pd.DataFrame(r["cm"], index=["真实·负面", "真实·正面"],
                         columns=["预测·负面", "预测·正面"]))
    display(pd.DataFrame(r["report"]).T.round(4))



===== [朴素贝叶斯] MultinomialNB =====


,预测·负面,预测·正面
真实·负面,1673,327
真实·正面,361,1639


,precision,recall,f1-score,support
负面,0.8225,0.8365,0.8294,2000.000
正面,0.8337,0.8195,0.8265,2000.000
accuracy,0.8280,0.8280,0.8280,0.828
macro avg,0.8281,0.8280,0.8280,4000.000
weighted avg,0.8281,0.8280,0.8280,4000.000



===== [朴素贝叶斯] ComplementNB =====


,预测·负面,预测·正面
真实·负面,1673,327
真实·正面,361,1639


...

,预测·负面,预测·正面
真实·负面,1651,349
真实·正面,401,1599


,precision,recall,f1-score,support
负面,0.8046,0.8255,0.8149,2000.0000
正面,0.8208,0.7995,0.8100,2000.0000
accuracy,0.8125,0.8125,0.8125,0.8125
macro avg,0.8127,0.8125,0.8125,4000.0000
weighted avg,0.8127,0.8125,0.8125,4000.0000



===== [多层感知机] MLPClassifier =====


,预测·负面,预测·正面
真实·负面,1651,349
真实·正面,323,1677


,precision,recall,f1-score,support
负面,0.8364,0.8255,0.8309,2000.000
正面,0.8277,0.8385,0.8331,2000.000
accuracy,0.8320,0.8320,0.8320,0.832
macro avg,0.8321,0.8320,0.8320,4000.000
weighted avg,0.8321,0.8320,0.8320,4000.000


## 9 实验结果

汇总全部15种方法的实验结果，列明文本表示、特征处理、分类器配置与验证指标，在报告中比较并分析实际结果。

保存实验相关文件与结果。具体提交要求见实验指导书。


In [11]:
# 实验结果汇总：列明统一文本表示、特征处理、分类器配置与验证指标。
svd_map = {r["spec"]["key"]:
           "TruncatedSVD(300)" if r["spec"].get("needs_svd") else "原始稀疏特征"
           for r in method_results}
summary_df = result_df.copy()
summary_df.insert(0, "文本表示", f"{REPR_NAME}（{REPR_NOTES[REPR_NAME]}，TF-IDF）")
summary_df.insert(3, "特征处理", summary_df["key"].map(svd_map))
display(summary_df.drop(columns=["key"]))


,文本表示,方法,方法组,特征处理,Accuracy,Macro-F1,Precision(macro),Recall(macro),训练耗时(s),预测耗时(s),参数配置
5,char13（字符 1-3 gram，TF-IDF）,RidgeClassifier,线性分类,原始稀疏特征,0.84675,0.846712,0.847094,0.84675,0.1,0.0,alpha=1.0（L2 正则强度）
9,char13（字符 1-3 gram，TF-IDF）,SVC(RBF),距离与核方法,原始稀疏特征,0.84500,0.844981,0.845167,0.84500,174.9,46.1,"C=1.0, gamma=scale, kernel=rbf, cache=1GB"
4,char13（字符 1-3 gram，TF-IDF）,LinearSVC,线性分类,原始稀疏特征,0.84425,0.844221,0.844511,0.84425,0.3,0.0,C=1.0（间隔与错误的权衡）
3,char13（字符 1-3 gram，TF-IDF）,LogisticRegression,线性分类,原始稀疏特征,0.84000,0.839990,0.840087,0.84000,0.4,0.0,"C=1.0（L2 正则强度倒数）, max_iter=1000, lbfgs"
14,char13（字符 1-3 gram，TF-IDF）,MLPClassifier,多层感知机,TruncatedSVD(300),0.83200,0.831993,0.832056,0.83200,21.7,0.4,"TruncatedSVD(300)→隐层(256), relu, 内部早停, max_ite..."
0,char13（字符 1-3 gram，TF-IDF）,MultinomialNB,朴素贝叶斯,原始稀疏特征,0.82800,0.827988,0.828095,0.82800,0.0,0.0,alpha=1.0（拉普拉斯平滑）；输入为 TF-IDF 非负权重
1,char13（字符 1-3 gram，TF-IDF）,ComplementNB,朴素贝叶斯,原始稀疏特征,0.82800,0.827988,0.828095,0.82800,0.0,0.0,alpha=1.0；用互补类统计构造权重，缓解类别不平衡
11,char13（字符 1-3 gram，TF-IDF）,RandomForestClassifier,决策树与集成,原始稀疏特征,0.81950,0.819500,0.819501,0.81950,8.9,0.2,"n_estimators=200, max_features=sqrt（默认）, 并行"
12,char13（字符 1-3 gram，TF-IDF）,AdaBoostClassifier,决策树与集成,原始稀疏特征,0.81950,0.819488,0.819582,0.81950,38.8,0.5,"n_estimators=200, learning_rate=1.0, 基学习器为深度1决策树"
6,char13（字符 1-3 gram，TF-IDF）,Perceptron,线性分类,原始稀疏特征,0.81475,0.814513,0.816367,0.81475,0.0,0.0,"max_iter=1000, tol=1e-3（误分类驱动迭代更新）"


In [12]:
# 保存实验相关文件与结果（提交物）：
#   文本转换器 / 特征矩阵 / 特征词表、每方法预测明细与训练好的模型、
#   指标汇总表 / 分类报告 / 全方法预测宽表 / 最优方法记录 —— 均存入 classical/。
import gzip

import joblib
from scipy import sparse

joblib.dump(vectorizer, SAVE_DIR / f"vectorizer_{REPR_NAME}.joblib")
sparse.save_npz(SAVE_DIR / "X_train.npz", X_train)
sparse.save_npz(SAVE_DIR / "X_validation.npz", X_dev)
with gzip.open(SAVE_DIR / "feature_names.txt.gz", "wt", encoding="utf-8") as fh:
    fh.write("\n".join(vectorizer.get_feature_names_out()))
print(f"[特征文件] vectorizer + X_train.npz + X_validation.npz + feature_names"
      f"（{len(vectorizer.get_feature_names_out())} 维）已保存至 classical/")

for r in method_results:
    key = r["spec"]["key"]
    pd.DataFrame({
        "id": validation_df["id"], "text": validation_df["text"], "label": y_dev,
        "pred": r["pred"], "score_positive": r["scores"],
        "correct": (r["pred"] == y_dev).astype(int),
    }).to_csv(SAVE_DIR / "preds" / f"{key}.csv", index=False, encoding="utf-8-sig")
    joblib.dump(r["pipe"], SAVE_DIR / "models" / f"{key}.joblib")
print(f"[预测与模型] preds/（{len(method_results)} 个方法预测明细）+ "
      f"models/（{len(method_results)} 个模型）已保存")

wide = validation_df[["id", "text", "label"]].copy()
for r in method_results:
    wide[r["spec"]["key"]] = r["pred"]
wide.to_csv(SAVE_DIR / "validation_predictions.csv", index=False, encoding="utf-8-sig")

result_df.to_csv(SAVE_DIR / "classical15_results.csv", index=False, encoding="utf-8-sig")
save_json({r["spec"]["key"]: {"name_cn": r["spec"]["name_cn"],
                              "group": r["spec"]["group"],
                              "metrics": r["metrics"],
                              "classification_report": r["report"],
                              "confusion_matrix": r["cm"].tolist()}
           for r in method_results}, SAVE_DIR / "classical15_reports.json")
save_json({"best_method": best_row["方法"], "key": best_row["key"],
           "macro_f1": float(best_row["Macro-F1"]),
           "accuracy": float(best_row["Accuracy"]), "representation": REPR_NAME},
          SAVE_DIR / "classical15_best.json")
print("[提交物] validation_predictions.csv（全方法预测宽表）/ classical15_results.csv "
      "/ classical15_reports.json / classical15_best.json 已保存")


[特征文件] vectorizer + X_train.npz + X_validation.npz + feature_names（172914 维）已保存至 classical/
[预测与模型] preds/（15 个方法预测明细）+ models/（15 个模型）已保存
[结果] 已保存: classical/classical15_reports.json
[结果] 已保存: classical/classical15_best.json
[提交物] validation_predictions.csv（全方法预测宽表）/ classical15_results.csv / classical15_reports.json / classical15_best.json 已保存


## 10 改进对照与改进最优管线

对应指导书 4.2 第二段：“可调整词级或字符级 n-gram、TF-IDF 设置、特征处理及分类器参数，尝试合理方法改善结果。在报告中说明改动理由、配置及对照结果。”

按“单因素对照”原则分四组实验：A 特征空间（n-gram 范围 / min_df / 词字拼接）、B TF-IDF 设置与卡方特征选择、C 距离类方法的稀疏与 TruncatedSVD 稠密特征、D 分类器关键参数扫描。各组胜者再组合成候选管线，在验证集上逐一重算，取 Macro-F1 最高者为改进最优经典管线。

说明：D 组与最终组合含多次 RBF SVC 训练（单次 3—6 分钟），第 10 节整体约需 25 分钟。


In [13]:
# A. 特征空间对照：n-gram 范围、min_df 与词字拼接（统一 LR(C=10) 探针）。
# 改动理由：更长 n-gram 保留更多局部组合；min_df=1 保留低频特征；
# 词字拼接同时利用字符组合信息与分词后的词义单元。
refine_a_rows = []
for label, builder in [
    ("char13(基线)", lambda: build_vectorizer("char13")),
    ("char14", lambda: build_vectorizer("char14")),
    ("char13+min_df=1", lambda: build_vectorizer("char13", min_df=1)),
    ("词字拼接char13+word12", build_union_vectorizer),
]:
    set_seed()
    vec = builder()
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(dev_texts)
    t0 = time.time()
    clf = LogisticRegression(C=10, max_iter=1000, random_state=SEED)
    clf.fit(X_tr, y_train)
    pred = clf.predict(X_va)
    total_sec = time.time() - t0
    m = compute_metrics(y_dev, pred)
    refine_a_rows.append({"配置": label, "特征维度": X_tr.shape[1],
                          "Accuracy": m["accuracy"], "Macro-F1": m["macro_f1"],
                          "Precision(macro)": m["precision_macro"],
                          "Recall(macro)": m["recall_macro"],
                          "转换+训练+预测耗时(s)": round(total_sec, 1)})
    print(f"[A] {label}: F1={m['macro_f1']:.4f} 维度={X_tr.shape[1]}")

df_a = pd.DataFrame(refine_a_rows)
df_a.to_csv(SAVE_DIR / "refine_A_feature_space.csv", index=False, encoding="utf-8-sig")
display(df_a)


[A] char13(基线): F1=0.8442 维度=172914
[A] char14: F1=0.8440 维度=255232
[A] char13+min_df=1: F1=0.8460 维度=614496
[A] 词字拼接char13+word12: F1=0.8437 维度=245320


,配置,特征维度,Accuracy,Macro-F1,Precision(macro),Recall(macro),转换+训练+预测耗时(s)
0,char13(基线),172914,0.84425,0.844223,0.844492,0.84425,1.4
1,char14,255232,0.84400,0.843972,0.844251,0.84400,1.8
2,char13+min_df=1,614496,0.84600,0.845961,0.846355,0.84600,2.5
3,词字拼接char13+word12,245320,0.84375,0.843734,0.843895,0.84375,1.5


In [14]:
# B. TF-IDF 设置与特征选择（统一 char13 表示上用 LR(C=10) 探针）。
# 改动理由：sublinear_tf 抑制高频词项；min_df 压缩低频噪声；
# 卡方检验只保留与类别关联最强的前 k 个特征。
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline

refine_b_rows = []
for label, vec, note in [
    ("char13 基线", build_vectorizer("char13"), "sublinear_tf=True, min_df=2"),
    ("sublinear_tf=False", build_vectorizer("char13", sublinear_tf=False),
     "线性词频权重（对照项）"),
    ("min_df=5", build_vectorizer("char13", min_df=5), "更激进的低频过滤"),
]:
    set_seed()
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(dev_texts)
    clf = LogisticRegression(C=10, max_iter=1000, random_state=SEED)
    clf.fit(X_tr, y_train)
    pred = clf.predict(X_va)
    m = compute_metrics(y_dev, pred)
    refine_b_rows.append({"配置": label, "说明": note, "特征维度": X_tr.shape[1],
                          "Accuracy": m["accuracy"], "Macro-F1": m["macro_f1"],
                          "Precision(macro)": m["precision_macro"],
                          "Recall(macro)": m["recall_macro"]})
    print(f"[B] {label}: F1={m['macro_f1']:.4f}")

for k in [100_000, 50_000]:   # 卡方特征选择：保留类别关联最强的前 k 个特征
    set_seed()
    vec = build_vectorizer("char13")
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(dev_texts)
    pipe = Pipeline([("chi2", SelectKBest(chi2, k=k)),
                     ("clf", LogisticRegression(C=10, max_iter=1000,
                                                random_state=SEED))])
    pipe.fit(X_tr, y_train)
    pred = pipe.predict(X_va)
    m = compute_metrics(y_dev, pred)
    refine_b_rows.append({"配置": f"卡方选择top{k}",
                          "说明": f"chi2 保留关联最强 {k} 特征", "特征维度": k,
                          "Accuracy": m["accuracy"], "Macro-F1": m["macro_f1"],
                          "Precision(macro)": m["precision_macro"],
                          "Recall(macro)": m["recall_macro"]})
    print(f"[B] chi2 top{k}: F1={m['macro_f1']:.4f}")

df_b = pd.DataFrame(refine_b_rows)
df_b.to_csv(SAVE_DIR / "refine_B_tfidf_settings.csv", index=False, encoding="utf-8-sig")
display(df_b)


[B] char13 基线: F1=0.8442
[B] sublinear_tf=False: F1=0.8407
[B] min_df=5: F1=0.8392
[B] chi2 top100000: F1=0.8432
[B] chi2 top50000: F1=0.8457


,配置,说明,特征维度,Accuracy,Macro-F1,Precision(macro),Recall(macro)
0,char13 基线,"sublinear_tf=True, min_df=2",172914,0.84425,0.844223,0.844492,0.84425
1,sublinear_tf=False,线性词频权重（对照项）,172914,0.84075,0.840710,0.841088,0.84075
2,min_df=5,更激进的低频过滤,54512,0.83925,0.839239,0.839342,0.83925
3,卡方选择top100000,chi2 保留关联最强 100000 特征,100000,0.84325,0.843242,0.843322,0.84325
4,卡方选择top50000,chi2 保留关联最强 50000 特征,50000,0.84575,0.845743,0.845813,0.84575


In [15]:
# C. 特征处理对照（面向距离/核方法）：KNN 在稀疏 TF-IDF vs TruncatedSVD 稠密特征。
# 改动理由：稠密低维空间近邻搜索更稳健、预测更快；另对照 LR 在 SVD 表示上的表现。
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier

refine_c_rows = []

def run_c(feature_name, clf_name, clf, X_tr, X_va):
    """训练 + 预测一次，记录指标与预测耗时。"""
    set_seed()
    clf.fit(X_tr, y_train)
    t0 = time.time()
    pred = clf.predict(X_va)
    pred_sec = time.time() - t0
    m = compute_metrics(y_dev, pred)
    refine_c_rows.append({"特征处理": feature_name, "分类器": clf_name,
                          "Accuracy": m["accuracy"], "Macro-F1": m["macro_f1"],
                          "Precision(macro)": m["precision_macro"],
                          "Recall(macro)": m["recall_macro"],
                          "预测耗时(s)": round(pred_sec, 1)})
    print(f"[C] {feature_name} + {clf_name}: F1={m['macro_f1']:.4f}")


# C1：直接使用 L2 归一化稀疏 TF-IDF（欧氏距离≡余弦排序）
run_c("稀疏 TF-IDF(20万+维)", "KNN(k=15,distance)",
      KNeighborsClassifier(n_neighbors=15, weights="distance", metric="euclidean",
                           algorithm="brute", n_jobs=-1), X_train, X_dev)

# C2：TruncatedSVD(300) 稠密化后 KNN
set_seed()
svd = TruncatedSVD(n_components=300, random_state=SEED)
X_svd = svd.fit_transform(X_train)
X_dev_svd = svd.transform(X_dev)
run_c("TruncatedSVD(300)稠密", "KNN(k=15,distance)",
      KNeighborsClassifier(n_neighbors=15, weights="distance", metric="euclidean",
                           n_jobs=-1), X_svd, X_dev_svd)

# C3：SVD 稠密 + 逻辑回归（对照线性模型在降维表示上的表现）
set_seed()
svd = TruncatedSVD(n_components=300, random_state=SEED)
X_svd = svd.fit_transform(X_train)
X_dev_svd = svd.transform(X_dev)
run_c("TruncatedSVD(300)稠密", "LogisticRegression(C=10)",
      LogisticRegression(C=10, max_iter=2000, random_state=SEED), X_svd, X_dev_svd)

df_c = pd.DataFrame(refine_c_rows)
df_c.to_csv(SAVE_DIR / "refine_C_feature_process.csv", index=False, encoding="utf-8-sig")
display(df_c)


[C] 稀疏 TF-IDF(20万+维) + KNN(k=15,distance): F1=0.7805
[C] TruncatedSVD(300)稠密 + KNN(k=15,distance): F1=0.7330
[C] TruncatedSVD(300)稠密 + LogisticRegression(C=10): F1=0.8307


,特征处理,分类器,Accuracy,Macro-F1,Precision(macro),Recall(macro),预测耗时(s)
0,稀疏 TF-IDF(20万+维),"KNN(k=15,distance)",0.78075,0.780477,0.782152,0.78075,2.0
1,TruncatedSVD(300)稠密,"KNN(k=15,distance)",0.73575,0.733040,0.745727,0.73575,0.2
2,TruncatedSVD(300)稠密,LogisticRegression(C=10),0.83075,0.830748,0.830769,0.83075,0.0


In [16]:
# D. 分类器关键参数扫描（统一在 char13 稀疏特征上；每组只改一个参数）。
# 改动理由：正则强度（C / alpha）、邻居数（k）直接决定模型复杂度与泛化能力。
# 注意：包含 2 次 RBF SVC 训练（受训练时间限制仅取 C=0.3 与 C=3.0），
# 本节约需 8—10 分钟。

def classifier_zoo():
    """（标签 → 分类器构造器）映射；最终候选组合复用同一来源，避免配置漂移。"""
    zoo = {}
    for C in [0.5, 1.0, 5.0, 20.0]:
        zoo[f"LogisticRegression C={C}"] = (
            lambda C=C: LogisticRegression(C=C, max_iter=1000, random_state=SEED))
    for C in [0.1, 0.5, 2.0]:
        zoo[f"LinearSVC C={C}"] = (
            lambda C=C: LinearSVC(C=C, max_iter=5000, random_state=SEED))
    for alpha in [0.1, 0.3, 3.0]:
        zoo[f"MultinomialNB alpha={alpha}"] = (
            lambda a=alpha: MultinomialNB(alpha=a))
    for k in [5, 30, 50]:
        zoo[f"KNN k={k}"] = (
            lambda k=k: KNeighborsClassifier(n_neighbors=k, weights="distance",
                                             metric="euclidean", algorithm="brute",
                                             n_jobs=-1))
    for C in [0.3, 3.0]:
        zoo[f"SVC(RBF) C={C}"] = (
            lambda C=C: SVC(C=C, kernel="rbf", gamma="scale", cache_size=1024,
                            random_state=SEED))
    return zoo


refine_d_rows = []
for label, make_clf in classifier_zoo().items():
    set_seed()
    t0 = time.time()
    clf = make_clf()
    clf.fit(X_train, y_train)
    fit_sec = time.time() - t0
    pred = clf.predict(X_dev)
    m = compute_metrics(y_dev, pred)
    refine_d_rows.append({"方法": label, "Accuracy": m["accuracy"],
                          "Macro-F1": m["macro_f1"],
                          "Precision(macro)": m["precision_macro"],
                          "Recall(macro)": m["recall_macro"],
                          "fit_time": round(fit_sec, 1)})
    print(f"[D] {label}: F1={m['macro_f1']:.4f} ({fit_sec:.1f}s)", flush=True)

df_d = pd.DataFrame(refine_d_rows)
df_d.to_csv(SAVE_DIR / "refine_D_params.csv", index=False, encoding="utf-8-sig")
display(df_d)


[D] LogisticRegression C=0.5: F1=0.8350 (0.5s)
[D] LogisticRegression C=1.0: F1=0.8400 (0.3s)
[D] LogisticRegression C=5.0: F1=0.8465 (0.5s)
[D] LogisticRegression C=20.0: F1=0.8410 (0.7s)
[D] LinearSVC C=0.1: F1=0.8425 (0.1s)
[D] LinearSVC C=0.5: F1=0.8487 (0.2s)
[D] LinearSVC C=2.0: F1=0.8382 (0.4s)
[D] MultinomialNB alpha=0.1: F1=0.8295 (0.0s)
[D] MultinomialNB alpha=0.3: F1=0.8280 (0.0s)
[D] MultinomialNB alpha=3.0: F1=0.8229 (0.0s)
[D] KNN k=5: F1=0.7474 (0.0s)
[D] KNN k=30: F1=0.7989 (0.0s)
[D] KNN k=50: F1=0.8112 (0.0s)
[D] SVC(RBF) C=0.3: F1=0.8292 (133.6s)
[D] SVC(RBF) C=3.0: F1=0.8492 (315.9s)


,方法,Accuracy,Macro-F1,Precision(macro),Recall(macro),fit_time
0,LogisticRegression C=0.5,0.83500,0.834983,0.835134,0.83500,0.5
1,LogisticRegression C=1.0,0.84000,0.839990,0.840087,0.84000,0.3
2,LogisticRegression C=5.0,0.84650,0.846474,0.846734,0.84650,0.5
3,LogisticRegression C=20.0,0.84100,0.840964,0.841307,0.84100,0.7
4,LinearSVC C=0.1,0.84250,0.842487,0.842611,0.84250,0.1
5,LinearSVC C=0.5,0.84875,0.848721,0.849014,0.84875,0.2
6,LinearSVC C=2.0,0.83825,0.838224,0.838470,0.83825,0.4
7,MultinomialNB alpha=0.1,0.82950,0.829500,0.829501,0.82950,0.0
8,MultinomialNB alpha=0.3,0.82800,0.828000,0.828001,0.82800,0.0
9,MultinomialNB alpha=3.0,0.82300,0.822929,0.823518,0.82300,0.0


In [17]:
# 最终改进管线：由 A/B/D 组胜者组合成候选管线，全部在验证集上重算，取 Macro-F1 最高者。
# 注意：其中含 3 次 RBF SVC 重训，本节约需 15—17 分钟，请耐心等待运行完毕。
a_best_label = df_a.loc[df_a["Macro-F1"].idxmax(), "配置"]
b_best_label = df_b.loc[df_b["Macro-F1"].idxmax(), "配置"]
d_best_label = df_d.loc[df_d["Macro-F1"].idxmax(), "方法"]
print(f"各组胜者：A={a_best_label} | B={b_best_label} | D={d_best_label}")

builders = {
    "char13(基线)": ("char13", lambda: build_vectorizer("char13")),
    "char14": ("char14", lambda: build_vectorizer("char14")),
    "char13+min_df=1": ("char13+min_df=1",
                        lambda: build_vectorizer("char13", min_df=1)),
    "词字拼接char13+word12": ("union", build_union_vectorizer),
}
a_key, a_make = builders[a_best_label]
base_key, base_make = builders["char13(基线)"]
d_make = classifier_zoo()[d_best_label]
probe = lambda: LogisticRegression(C=10, max_iter=1000, random_state=SEED)

# B 组胜者仅在“卡方特征选择”时参与组合（其余 B 配置均为被淘汰的对照项）
chi2_k = None
if b_best_label.startswith("卡方选择top"):
    chi2_k = int(b_best_label.replace("卡方选择top", ""))

# 候选列表：(标签, 向量器key, 向量器构造, chi2_k, 分类器标签, 分类器构造)
candidates = [
    ("探针基线：char13 + LR(C=10)", base_key, base_make, None, "LR(C=10)", probe),
    (f"A组最优表示：{a_best_label} + LR(C=10)", a_key, a_make, None, "LR(C=10)", probe),
    (f"D组最优分类器：char13 + {d_best_label}", base_key, base_make, None,
     d_best_label, d_make),
    (f"A+D 组合：{a_best_label} + {d_best_label}", a_key, a_make, None,
     d_best_label, d_make),
]
if chi2_k:
    candidates += [
        (f"B组最优表示：char13+卡方top{chi2_k} + LR(C=10)", base_key, base_make,
         chi2_k, "LR(C=10)", probe),
        (f"B+D 组合：char13+卡方top{chi2_k} + {d_best_label}", base_key, base_make,
         chi2_k, d_best_label, d_make),
        (f"A+B 组合：{a_best_label}+卡方top{chi2_k} + LR(C=10)", a_key, a_make,
         chi2_k, "LR(C=10)", probe),
    ]

final_rows, best = [], None
for label, vkey, vmake, k, clabel, cmake in candidates:
    set_seed()
    vec = vmake()
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(dev_texts)
    selector = None
    if k:
        selector = SelectKBest(chi2, k=k)
        X_tr = selector.fit_transform(X_tr, y_train)
        X_va = selector.transform(X_va)
    clf = cmake()
    t0 = time.time()
    clf.fit(X_tr, y_train)
    fit_sec = time.time() - t0
    pred = clf.predict(X_va)
    m = compute_metrics(y_dev, pred)
    final_rows.append({"候选配置": label, "特征维度": int(X_tr.shape[1]),
                       "Accuracy": m["accuracy"], "Macro-F1": m["macro_f1"],
                       "Precision(macro)": m["precision_macro"],
                       "Recall(macro)": m["recall_macro"],
                       "训练耗时(s)": round(fit_sec, 1)})
    print(f"[final] {label}: F1={m['macro_f1']:.4f} ({fit_sec:.1f}s)", flush=True)
    if best is None or m["macro_f1"] > best["metrics"]["macro_f1"]:
        best = {"label": label, "vec": vec, "selector": selector, "clf": clf,
                "X_dev": X_va, "metrics": m, "pred": pred,
                "signature": {"表示": vkey, "卡方topk": k, "分类器": clabel}}

df_final = pd.DataFrame(final_rows)
df_final.to_csv(SAVE_DIR / "refine_final_selection.csv", index=False,
                encoding="utf-8-sig")
display(df_final)

# 改进最优：保存为可直接复用的完整管线（TF-IDF → 卡方选择 → 分类器）与预测明细
steps = [("tfidf", best["vec"])]
if best["selector"] is not None:
    steps.append(("chi2", best["selector"]))
steps.append(("clf", best["clf"]))
joblib.dump(Pipeline(steps), SAVE_DIR / "models" / "best_refined.joblib")
if hasattr(best["clf"], "decision_function"):
    score = best["clf"].decision_function(best["X_dev"])
else:
    score = best["clf"].predict_proba(best["X_dev"])[:, 1]
pd.DataFrame({"id": validation_df["id"], "text": validation_df["text"],
              "label": y_dev, "pred": best["pred"], "score_positive": score,
              "correct": (best["pred"] == y_dev).astype(int)}).to_csv(
    SAVE_DIR / "preds" / "best_refined.csv", index=False, encoding="utf-8-sig")
baseline_15 = json.loads((SAVE_DIR / "classical15_best.json").read_text(encoding="utf-8"))
save_json({"method": best["label"], "signature": best["signature"],
           "metrics": best["metrics"],
           "confusion_matrix": confusion_matrix(y_dev, best["pred"],
                                                labels=[0, 1]).tolist(),
           "baseline_15": baseline_15}, SAVE_DIR / "refine_best.json")
print(f"改进最优：{best['label']}（验证 Macro-F1={best['metrics']['macro_f1']:.4f}）")


各组胜者：A=char13+min_df=1 | B=卡方选择top50000 | D=SVC(RBF) C=3.0
[final] 探针基线：char13 + LR(C=10): F1=0.8442 (1.4s)
[final] A组最优表示：char13+min_df=1 + LR(C=10): F1=0.8460 (2.4s)
[final] D组最优分类器：char13 + SVC(RBF) C=3.0: F1=0.8492 (297.6s)
[final] A+D 组合：char13+min_df=1 + SVC(RBF) C=3.0: F1=0.8480 (306.3s)
[final] B组最优表示：char13+卡方top50000 + LR(C=10): F1=0.8457 (1.1s)
[final] B+D 组合：char13+卡方top50000 + SVC(RBF) C=3.0: F1=0.8455 (199.8s)
[final] A+B 组合：char13+min_df=1+卡方top50000 + LR(C=10): F1=0.8495 (0.8s)


,候选配置,特征维度,Accuracy,Macro-F1,Precision(macro),Recall(macro),训练耗时(s)
0,探针基线：char13 + LR(C=10),172914,0.84425,0.844223,0.844492,0.84425,1.4
1,A组最优表示：char13+min_df=1 + LR(C=10),614496,0.84600,0.845961,0.846355,0.84600,2.4
2,D组最优分类器：char13 + SVC(RBF) C=3.0,172914,0.84925,0.849229,0.849443,0.84925,297.6
3,A+D 组合：char13+min_df=1 + SVC(RBF) C=3.0,614496,0.84800,0.847982,0.848169,0.84800,306.3
4,B组最优表示：char13+卡方top50000 + LR(C=10),50000,0.84575,0.845743,0.845813,0.84575,1.1
5,B+D 组合：char13+卡方top50000 + SVC(RBF) C=3.0,50000,0.84550,0.845491,0.845578,0.84550,199.8
6,A+B 组合：char13+min_df=1+卡方top50000 + LR(C=10),50000,0.84950,0.849489,0.849601,0.84950,0.8


[结果] 已保存: classical/refine_best.json
改进最优：A+B 组合：char13+min_df=1+卡方top50000 + LR(C=10)（验证 Macro-F1=0.8495）
